In [39]:
import os
import logging
import numpy as np
import scipy.io as sio
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

def download_seismic_data(
    start_date="2022-09-05",
    end_date="2022-09-06",
    network="2F",
    station_pattern="AX*",
    location_pattern="*",
    channel_patterns=["HH?", "EL?"],
    output_folder="./daily_mseed_files",
    username="mczhang8@uw.edu",
    token="RxL42LH6XTNFybHb"
):
    """
    Download seismic data from IRIS and save as mat and mseed files.

    Args:
        start_date (str): Start date in YYYY-MM-DD format
        end_date (str): End date in YYYY-MM-DD format
        network (str): Network code
        station_pattern (str): Station pattern (e.g., "AX*")
        location_pattern (str): Location pattern
        channel_patterns (list): List of channel patterns
        output_folder (str): Output directory for mseed files
        username (str): IRIS username
        token (str): IRIS authentication token

    Returns:
        dict: Trace metadata and data
    """
    # Initialize IRIS client
    try:
        client = Client("IRIS", user=username, password=token)
        logger.info("Initialized IRIS client")
    except Exception as e:
        logger.error(f"Failed to initialize IRIS client: {str(e)}")
        return {}

    # Create output folder
    os.makedirs(output_folder, exist_ok=True)
    logger.info(f"Output folder: {output_folder}")

    # Initialize data containers
    trace_dict = {
        'network': [], 'station': [], 'location': [], 'channel': [],
        'sensitivity': [], 'sensitivityFrequency': [], 'data': [],
        'sampleCount': [], 'sampleRate': [], 'startTime': [], 'endTime': []
    }

    # Convert dates
    try:
        start = UTCDateTime(start_date)
        end = UTCDateTime(end_date)
        logger.info(f"Processing date range: {start_date} to {end_date}")
    except Exception as e:
        logger.error(f"Invalid date format: {str(e)}")
        return {}

    # Get station inventory
    try:
        inventory = client.get_stations(
            network=network,
            station=station_pattern,
            location=location_pattern,
            channel=",".join(channel_patterns),
            starttime=start,
            endtime=end,
            level="channel"
        )
        logger.info(f"Found {len(inventory.networks)} networks, {sum(len(sta) for net in inventory.networks for sta in net.stations)} stations")
        for net in inventory.networks:
            for sta in net.stations:
                logger.info(f"Station {sta.code}: {len(sta.channels)} channels")
    except Exception as e:
        logger.error(f"Failed to retrieve inventory: {str(e)}")
        return trace_dict

    # Loop through each day
    current_date = start
    while current_date < end:
        next_date = min(current_date + 86400, end)
        date_str = current_date.strftime('%Y-%m-%d')
        logger.info(f"Processing {date_str}")

        for net in inventory.networks:
            for sta in net.stations:
                for chan in sta.channels:
                    try:
                        logger.info(f"Downloading {net.code}.{sta.code}.{chan.location_code}.{chan.code}")
                        stream = client.get_waveforms(
                            network=net.code,
                            station=sta.code,
                            location=chan.location_code,
                            channel=chan.code,
                            starttime=current_date,
                            endtime=next_date,
                            attach_response=True
                        )

                        if not stream:
                            logger.warning(f"No data for {net.code}.{sta.code}.{chan.location_code}.{chan.code}")
                            continue

                        # Save mseed file
                        mseed_file = os.path.join(
                            output_folder,
                            f"{net.code}.{sta.code}.{chan.location_code}.{chan.code}.{date_str}.mseed"
                        )
                        stream.write(mseed_file)
                        logger.info(f"Saved {mseed_file}")

                        # Process each trace
                        for trace in stream:
                            try:
                                resp = trace.stats.response._get_overall_sensitivity_and_gain()
                                trace_dict['network'].append(trace.stats.network)
                                trace_dict['station'].append(trace.stats.station)
                                trace_dict['location'].append(trace.stats.location)
                                trace_dict['channel'].append(trace.stats.channel)
                                trace_dict['sensitivity'].append(float(resp[1]))
                                trace_dict['sensitivityFrequency'].append(float(resp[0]))
                                trace_dict['data'].append(trace.data.astype(np.float64))
                                trace_dict['sampleCount'].append(int(trace.stats.npts))
                                trace_dict['sampleRate'].append(float(trace.stats.sampling_rate))
                                trace_dict['startTime'].append(trace.stats.starttime.strftime("%Y-%m-%d:%H:%M:%S.%f"))
                                trace_dict['endTime'].append(trace.stats.endtime.strftime("%Y-%m-%d:%H:%M:%S.%f"))
                                logger.info(f"Processed {trace.stats.network}.{trace.stats.station}.{trace.stats.channel} ({trace.stats.npts} samples)")
                            except Exception as e:
                                logger.error(f"Error processing trace {sta.code}.{chan.code}: {str(e)}")
                                continue

                    except Exception as e:
                        logger.warning(f"Error downloading {net.code}.{sta.code}.{chan.location_code}.{chan.code}: {str(e)}")
                        continue

        current_date = next_date

    # Save trace dictionary to mat file
    mat_file = os.path.join(output_folder, f"py_trace_{start_date}.mat")
    try:
        sio.savemat(mat_file, {'trace': trace_dict}, format='5', do_compression=True)
        file_size = os.path.getsize(mat_file) / 1024  # Size in KB
        logger.info(f"Saved {mat_file} ({file_size:.2f} KB), {len(trace_dict['data'])} traces")
    except Exception as e:
        logger.error(f"Failed to save {mat_file}: {str(e)}")

    return trace_dict

if __name__ == "__main__":
    trace_data = download_seismic_data()
    if trace_data['network']:
        logger.info("Data download complete.")
    else:
        logger.warning("No data downloaded.")

2025-05-20 09:29:38,216 - INFO - Initialized IRIS client
2025-05-20 09:29:38,217 - INFO - Output folder: ./daily_mseed_files
2025-05-20 09:29:38,218 - INFO - Processing date range: 2022-09-05 to 2022-09-06
2025-05-20 09:29:38,349 - INFO - Found 1 networks, 45 stations
2025-05-20 09:29:38,350 - INFO - Station AX01A: 3 channels
2025-05-20 09:29:38,350 - INFO - Station AX02A: 3 channels
2025-05-20 09:29:38,351 - INFO - Station AX03A: 3 channels
2025-05-20 09:29:38,351 - INFO - Station AX04A: 3 channels
2025-05-20 09:29:38,352 - INFO - Station AX05A: 3 channels
2025-05-20 09:29:38,352 - INFO - Station AX06A: 3 channels
2025-05-20 09:29:38,353 - INFO - Station AX07A: 3 channels
2025-05-20 09:29:38,353 - INFO - Station AX08A: 3 channels
2025-05-20 09:29:38,354 - INFO - Station AX09A: 3 channels
2025-05-20 09:29:38,354 - INFO - Station AX10A: 3 channels
2025-05-20 09:29:38,355 - INFO - Station AX11A: 3 channels
2025-05-20 09:29:38,355 - INFO - Station AX12A: 3 channels
2025-05-20 09:29:38,356